# Spam classifier training

Load and preprocess the SMS dataset, train a text-classification pipeline, and save the model artifact used by `app.py`.

In [ ]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "spam.csv"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
MODEL_PATH = ARTIFACT_DIR / "spam_classifier.pkl"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Run this notebook from the project root.")

## Preprocess the dataset

Keep the label and message columns, give them descriptive names, and lowercase messages to match the app's input normalization.

In [ ]:
data = pd.read_csv(DATA_PATH, encoding="latin-1")[["v1", "v2"]]
data.columns = ["label", "text"]
data["text"] = data["text"].str.lower()

data.head()

## Evaluate the classifier

Use a stratified 70/30 train/test split to evaluate performance on messages not used to fit the evaluation pipeline. Precision, recall, and F1 are macro averages across the `ham` and `spam` classes, weighting both classes equally. Accuracy is the overall fraction of correct predictions.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.3,
    random_state=42,
    stratify=data["label"],
)

evaluation_model = make_pipeline(CountVectorizer(), MultinomialNB())
evaluation_model.fit(X_train, y_train)
y_pred = evaluation_model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro-average precision: {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Macro-average recall:    {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Macro-average F1 score:  {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

print("Confusion matrix (rows: actual, columns: predicted; ham, spam):")
confusion_matrix(y_test, y_pred, labels=["ham", "spam"])

Accuracy:  0.9839
Macro-average precision: 0.9766
Macro-average recall:    0.9529
Macro-average F1 score:  0.9643
Confusion matrix (rows: actual, columns: predicted; ham, spam):


array([[1441,    7],
       [  20,  204]])

## Train and save the final model

After evaluation, refit the pipeline on the full dataset to make use of all labeled messages. The saved pipeline keeps vectorization and classification together so the app can predict directly from messages.

In [5]:
model = make_pipeline(CountVectorizer(), MultinomialNB())
model.fit(data["text"], data["label"])

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(model, model_file)

print(f"Saved trained model to {MODEL_PATH}")

Saved trained model to c:\Users\Aaron\Documents\Data Science\spam\Spam-Classification\artifacts\spam_classifier.pkl
